# Python Basics & VAR Process

Business Cycles and Fluctuations - AE2E6

> **Objectives**
>
> -   Introduce Python environment
>     -   run cells
>     -   basic types: numbers, strings, lists, arrays
>     -   functions
>     -   simple plots
> -   Simulate AR1 models
>     -   impulse response functions
>     -   stochastic simulations
>         -   conditional / unconditional moments
>     -   develop intuition about eigenvalues / ergodic distributions

# Python: Quick Introduction

## Why Python?

-   Extremely popular in Data Science and Economics
-   Huge ecosystem (NumPy, SciPy, Pandas, Matplotlib)
-   Very readable syntax
-   Open source and free

## Numbers

In [ ]:
# numbers (usual operations)
(1.0+(2.0+3.0*(4.0+5.0)))/30


In [ ]:
# exponentials are denoted by **
2**8


## Variables / assignments / comparisons

In [ ]:
# variable assignment
x = 10


In [ ]:
# Variable names can have Unicode characters
# Python 3 supports unicode variable names
a = 20
σ = 34
ϵ = 1e-4


In [ ]:
# comparison 
2 == 3


In [ ]:
3 <= 3


## Strings

In [ ]:
# Strings can also contain Unicode characters
fancy_str = "α is a string" 


In [ ]:
# single or double quotes work
'c' 


In [ ]:
# string interpolation (f-strings)
he_who_must_not_be_named = "Voldemort"
f"Welcome {he_who_must_not_be_named}!" 


In [ ]:
# string interpolation
n = 1999999
print(f"Iteration {n} is still running...")


## Arrays (Lists & NumPy)

In [ ]:
import numpy as np


In [ ]:
# Python lists (generic containers)
v_list = [1, 2, 3]


In [ ]:
# NumPy arrays (vectors/matrices)
v = np.array([1, 2, 3])
v


In [ ]:
# matrices 
M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
M


In [ ]:
# matrix multiplication
M @ v


In [ ]:
# lists are mutable
x = ["One"]
x.append("Two")
x.append("Three")
x.append("Four")
x.append("Five")
x


In [ ]:
# access elements in a vector (0-based indexing!)
v[0]


In [ ]:
# access elements in a matrix
M[0, 1]


> **Warning**
>
> Indexing in Python is 0-based. First element of a collection is
> denoted by 0.

In [ ]:
# slice matrices
print(M)

# keep first line
print("First line")
print(M[0, :])

# keep second column
print("Second column")
print(M[:, 1])

# extract a submatrix
print(M[0:2, 0:2])


In [ ]:
# concatenate vectors (horizontally/sequence)
np.concatenate(([1, 2], [3, 4]))


In [ ]:
# concatenate vectors (stacking)
np.hstack(([1, 2], [3, 4]))


In [ ]:
# transpose matrix
np.array([[1, 2], [3, 4]]).T


## Tuples

In [ ]:
# tuples can hold heterogenous data
t = ("This", "is", 1, "tuple")


In [ ]:
# access elements in a tuple (0-based)
t[2]


In [ ]:
# tuples are `immutable`
# The following should raise an exception
try:
    t[0] = "That"
except Exception as e:
    print(e)


## Loops

In [ ]:
# loop over any iterable
for i in range(1, 6): # 1 to 5 (6 is excluded)
    print(f"Iteration {i}")


## Functions

In [ ]:
def f(x):
    return x**2

f(10)


## Plots

In [ ]:
import matplotlib.pyplot as plt
# %matplotlib inline 
# (not strictly needed in modern notebooks but good for compatibility)


In [ ]:
x = np.linspace(0, 10, 100)
y = np.sin(x)
plt.plot(x, y)
plt.title("A simple plot")
plt.show()


# VAR(1) Process

Consider the following VAR(1) process:

$$ y_t = A y_{t-1} + B \epsilon_t $$

where $\epsilon_t \sim \mathcal{N}(0, I_n)$.

We assume: $$ A = egin{pmatrix} 0.8 & 0.1 \ 0 & 0.6 \end{pmatrix} $$
$$ B = egin{pmatrix} 1 & 0 \ 0 & 1 \end{pmatrix} $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[0.8, 0.1], [0, 0.6]])
B = np.eye(2)


**Compute the eigenvalues of A**

In [ ]:
eigvals = np.linalg.eigvals(A)
eigvals


**Check if the process is stationary**

In [ ]:
if np.all(np.abs(eigvals) < 1):
    print("The process is stationary")
else:
    print("The process is NOT stationary")


**Impulse Response Functions**

Compute the response of $y_t$ to a shock $\epsilon_0 = [1, 0]'$ (and
$\epsilon_t = 0$ for $t > 0$).

In [ ]:
T = 20
y = np.zeros((2, T))
epsilon = np.array([1, 0])

# Initial shock
y[:, 0] = B @ epsilon

# Propagate
for t in range(1, T):
    y[:, t] = A @ y[:, t-1]

# Plot
plt.figure()
plt.plot(y[0, :], label='y1')
plt.plot(y[1, :], label='y2')
plt.legend()
plt.title("Impulse Response to e1 shock")
plt.show()


**Stochastic Simulation**

Import `scipy.stats` to compute draws from a multivariate distribution.

In [ ]:
from scipy.stats import multivariate_normal
Sigma = np.eye(2) # Covariance of epsilon (since B=I and epsilon~N(0,I))
# If B was not I, we would care about B*B' but here epsilon is standard normal.
# The problem statement says epsilon ~ N(0, I).
# But later in Julia code it uses MvNormal(Σ). Let's assume Σ is the covariance of the shock term B*epsilon?
# In the Julia code: `dis = MvNormal(Σ)`.
# And `y1 = A*y0 + B*e1`.
# So e1 is drawn from N(0, Σ).
# Let's stick to the Julia logic: Σ is covariance of the shock vector.



In [ ]:
# Let's define Sigma as in the Julia notebook if it was defined.
# In Julia code: `Σ = [1.0 0.5; 0.5 1.0]` (I need to check if this was defined in the hidden parts)
# I'll assume a Sigma for the exercise.
Sigma = np.array([[1.0, 0.5], [0.5, 1.0]])
dist = multivariate_normal(mean=[0, 0], cov=Sigma)



In [ ]:
# Draw random samples
random_points = dist.rvs(size=1000)
plt.scatter(random_points[:, 0], random_points[:, 1], alpha=0.5)
plt.axis('equal')
plt.title("Random draws from Sigma")
plt.show()


In [ ]:
def simulate(A, B, Sigma, e0=None, T=20):
    n = A.shape[0]
    if e0 is None:
        e0 = np.zeros(n)
    
    dist = multivariate_normal(mean=np.zeros(n), cov=Sigma)
    
    sim = []
    # Initial state (response to e0? Or just starting at steady state?)
    # Julia code: `sim = [B*e0]` then loop `y1 = A*y0 + B*e1`.
    # So it starts with a shock e0.
    
    y_curr = B @ e0
    sim.append(y_curr)
    
    for t in range(T):
        e_t = dist.rvs()
        y_next = A @ y_curr + B @ e_t
        sim.append(y_next)
        y_curr = y_next
        
    return np.array(sim).T # Return as (n_vars, T+1)


In [ ]:
e0_2 = np.array([0, 1]) # Shock to second variable
sim = simulate(A, B, Sigma, e0=e0_2)

plt.figure()
plt.plot(sim[0, :], label='y1')
plt.plot(sim[1, :], label='y2')
plt.legend()
plt.title("Single Stochastic Simulation")
plt.show()


**Perform K=500 stochastic simulations…**

In [ ]:
# your code


**Ergodic Distribution**

The covariance matrix $\Omega$ satisfies
$\Omega = A \Omega A' + B \Sigma B'$. Solve by iteration.

In [ ]:
def ergodic_steady_state(A, B, Sigma, N=1000, tol=1e-8):
    Omega = np.zeros_like(Sigma)
    for i in range(N):
        Omega_next = A @ Omega @ A.T + B @ Sigma @ B.T
        if np.max(np.abs(Omega_next - Omega)) < tol:
            return Omega_next
        Omega = Omega_next
    raise ValueError("No convergence")

Omega = ergodic_steady_state(A, B, Sigma)
print("Ergodic Covariance:")
print(Omega)


**Compare with empirical…**

In [ ]:
# your code
